In [1]:
!pip -q install sympy

In [2]:
from __future__ import annotations
from dataclasses import dataclass
from fractions import Fraction
from typing import Union, List, Dict, Tuple, Callable, Optional, Iterable, Set
import itertools
import math
import sympy as sp

# Hard

## Главные идеалы в $\mathbb Z$ и алгоритм Евклида

В этом разделе реализуем базовый класс `RingElement` для двух колец: $\mathbb Z$ и $K[x]$, а также функцию
`gcd_ring_elements`, возвращающую порождающий элемент главного идеала, порождённого заданными элементами.

Затем объясним критерий принадлежности $f\in (a_1,\dots,a_k)$:

- в $\mathbb Z$ идеал $(a_1,\dots,a_k)$ равен $(d)$, где $d=\gcd(a_1,\dots,a_k)$, поэтому $f\in (a_1,\dots,a_k) \iff d\mid f$;
- в $K[x]$ (где $K$ — поле) имеем $(f_1,\dots,f_k)=(g)$, где $g$ — НОД полиномов (берём монический), поэтому $f\in (f_1,\dots,f_k) \iff g\mid f$.

## Задание Hard 1. Идеалы в $\mathbb Z$

Нужно реализовать поиск порождающего элемента идеала в $\mathbb Z$ (то есть НОД целых чисел) и использовать его
для проверки принадлежности идеалу.

## Задание Hard 2. Идеалы в $K[x]$

Нужно реализовать поиск порождающего элемента идеала в $K[x]$ (то есть НОД полиномов) с нормировкой до монического полинома.


In [3]:
from typing import Union, List
class RingElement:
    """Базовый класс для элементов колец: целых чисел и полиномов."""
    def __init__(self, data: Union[int, List[float]]):
        """
        data:
        - если это целое число → элемент кольца Z;
        - если это список коэффициентов [a0, a1, ..., an] → полином a0 + a1*x + ... +
        ,→ an*x^n.
        """
        self.data = data
        self.is_polynomial = isinstance(data, list)
    def __repr__(self) -> str:
        if self.is_polynomial:
            terms = [
                f"{c}*x^{i}" if i > 0 else str(c)
                for i, c in enumerate(self.data)
                if c != 0
            ]
            return " + ".join(terms) if terms else "0"
        return str(self.data)
def _gcd_int(a: int, b: int) -> int:
    a, b = abs(a), abs(b)
    while b != 0:
        a, b = b, a % b
    return a
def _gcd_int_list(nums: List[int]) -> int:
    g = 0
    for x in nums:
        g = _gcd_int(g, int(x))
    return g
def _trim_poly(a: List[Fraction]) -> List[Fraction]:
    a = list(a)
    while len(a) > 0 and a[-1] == 0:
        a.pop()
    return a
def _poly_deg(a: List[Fraction]) -> int:
    a = _trim_poly(a)
    return len(a) - 1
def _poly_lc(a: List[Fraction]) -> Fraction:
    a = _trim_poly(a)
    return a[-1] if a else Fraction(0)
def _poly_add(a: List[Fraction], b: List[Fraction]) -> List[Fraction]:
    n = max(len(a), len(b))
    res = [Fraction(0) for _ in range(n)]
    for i in range(n):
        if i < len(a):
            res[i] += a[i]
        if i < len(b):
            res[i] += b[i]
    return _trim_poly(res)
def _poly_sub(a: List[Fraction], b: List[Fraction]) -> List[Fraction]:
    n = max(len(a), len(b))
    res = [Fraction(0) for _ in range(n)]
    for i in range(n):
        if i < len(a):
            res[i] += a[i]
        if i < len(b):
            res[i] -= b[i]
    return _trim_poly(res)
def _poly_mul_scalar(a: List[Fraction], c: Fraction) -> List[Fraction]:
    return _trim_poly([ai * c for ai in a])
def _poly_mul_xk(a: List[Fraction], k: int) -> List[Fraction]:
    if not a:
        return []
    return [Fraction(0)] * k + list(a)
def _poly_divmod(f: List[Fraction], g: List[Fraction]) -> tuple[List[Fraction], List[Fraction]]:
    f = _trim_poly(f)
    g = _trim_poly(g)
    if not g:
        raise ZeroDivisionError("polynomial division by zero")
    if not f:
        return [], []
    df, dg = _poly_deg(f), _poly_deg(g)
    lc_g = _poly_lc(g)
    q = [Fraction(0)] * max(0, df - dg + 1)
    r = list(f)
    while r and _poly_deg(r) >= dg:
        dr = _poly_deg(r)
        lc_r = _poly_lc(r)
        t_deg = dr - dg
        t_coeff = lc_r / lc_g
        q[t_deg] += t_coeff
        subtract_term = _poly_mul_scalar(_poly_mul_xk(g, t_deg), t_coeff)
        r = _poly_sub(r, subtract_term)
    return _trim_poly(q), _trim_poly(r)
def _poly_gcd(a: List[Fraction], b: List[Fraction]) -> List[Fraction]:
    a = _trim_poly(a)
    b = _trim_poly(b)
    if not a:
        return _monic(b)
    if not b:
        return _monic(a)
    while b:
        _, r = _poly_divmod(a, b)
        a, b = b, r
    return _monic(a)
def _monic(f: List[Fraction]) -> List[Fraction]:
    f = _trim_poly(f)
    if not f:
        return []
    lc = _poly_lc(f)
    if lc == 0:
        return []
    return _poly_mul_scalar(f, Fraction(1, 1) / lc)
def _poly_gcd_list(polys: List[List[Fraction]]) -> List[Fraction]:
    g: List[Fraction] = []
    for p in polys:
        g = _poly_gcd(g, p)
    return g
def gcd_ring_elements(elements: List[RingElement]) -> RingElement:
    """
    Возвращает порождающий главного идеала, порождённого заданными элементами.
    - Для Z: НОД целых чисел.
    - Для K[x]: НОД полиномов (с коэффициентами в поле K).
    """
    if not elements:
        raise ValueError("elements must be non-empty")
    is_poly = elements[0].is_polynomial
    if any(e.is_polynomial != is_poly for e in elements):
        raise TypeError("all elements must be from the same ring (all ints or all polynomials)")
    if not is_poly:
        nums = [int(e.data) for e in elements]
        d = _gcd_int_list(nums)
        return RingElement(d)
    polys = []
    for e in elements:
        coeffs = [Fraction(c).limit_denominator() for c in e.data]
        polys.append(coeffs)
    g = _poly_gcd_list(polys)
    return RingElement([float(c) if c.denominator != 1 else int(c.numerator) for c in g])

In [4]:
assert gcd_ring_elements([RingElement(12), RingElement(18), RingElement(-30)]).data == 6
assert gcd_ring_elements([RingElement(0), RingElement(0), RingElement(5)]).data == 5
def in_ideal_Z(f: int, gens: List[int]) -> bool:
    d = gcd_ring_elements([RingElement(x) for x in gens]).data
    d = int(d)
    return d == 0 and f == 0 or (d != 0 and f % d == 0)
assert in_ideal_Z(24, [12, 18, -30]) is True
assert in_ideal_Z(25, [12, 18, -30]) is False
x = sp.Symbol('x')
def poly_list_to_sympy(a: List[float]) -> sp.Poly:
    expr = sum(sp.Rational(str(a[i])) * x**i for i in range(len(a)))
    return sp.Poly(expr, x, domain=sp.QQ)
f1 = RingElement([ -1, 0, 1 ])
f2 = RingElement([ -1, 1 ])
g = gcd_ring_elements([f1, f2]).data
assert poly_list_to_sympy(g).as_expr() == sp.Poly(x-1, x, domain=sp.QQ).as_expr()
print("Hard tests passed.")

Hard tests passed.


# Expert

##  мономиальные порядки, редукция, базис Грёбнера

Пусть $R=K[x_1,\dots,x_n]$, где $K$ — поле. Мономы имеют вид $x^\alpha = x_1^{\alpha_1}\cdots x_n^{\alpha_n}$, где $\alpha\in\mathbb Z_{\ge 0}^n$.

### Определение (мономиальный порядок)
Отношение $\prec$ на мономах называется **мономиальным порядком**, если:
1) транзитивность: $a\prec b,\ b\prec c \Rightarrow a\prec c$;
2) тотальность: для любых $a,b$ верно $a=b$ или $a\prec b$ или $b\prec a$;
3) совместимость с умножением: $a\prec b \Rightarrow ac\prec bc$ для любого монома $c$.

---

### Примеры порядков: lex и grlex
Пусть $\alpha,\beta\in\mathbb Z_{\ge 0}^n$.

**Lex (лексикографический):** $\alpha\prec_{\mathrm{lex}}\beta$, если в первом индексе $i$, где $\alpha_i\ne\beta_i$, выполнено $\alpha_i<\beta_i$.

**Grlex (градуированный лексикографический):** $\alpha\prec_{\mathrm{grlex}}\beta$, если либо $|\alpha|<|\beta|$, где $|\alpha|=\sum_i\alpha_i$, либо $|\alpha|=|\beta|$ и $\alpha\prec_{\mathrm{lex}}\beta$.

#### Проверка свойства (3) для lex
Пусть $\alpha\prec_{\mathrm{lex}}\beta$ и $\gamma\in\mathbb Z_{\ge 0}^n$. Возьмём первый индекс $i$, где $\alpha_i\ne\beta_i$.
Тогда $(\alpha+\gamma)_j=(\beta+\gamma)_j$ для $j<i$ и $(\alpha+\gamma)_i<(\beta+\gamma)_i$, значит $\alpha+\gamma\prec_{\mathrm{lex}}\beta+\gamma$, то есть $x^{\alpha}x^\gamma\prec x^\beta x^\gamma$.

#### Проверка свойства (3) для grlex
Если $|\alpha|<|\beta|$, то $|\alpha+\gamma|<|\beta+\gamma|$. Если $|\alpha|=|\beta|$ и $\alpha\prec_{\mathrm{lex}}\beta$, то по доказанному выше $\alpha+\gamma\prec_{\mathrm{lex}}\beta+\gamma$, а суммы степеней равны; значит $\alpha+\gamma\prec_{\mathrm{grlex}}\beta+\gamma$.

---

### Редукция и нормальная форма
Пусть задан набор $G=\{g_1,\dots,g_s\}\subset R$. Редукция полинома $f$ по $G$ — это итеративное вычитание кратных элементов $G$, устраняющее ведущий моном, когда он делится на ведущий моном некоторого $g_i$.

Если редукция дала $0$, то $f\in (G)$, потому что каждое преобразование имеет вид
$$
f \longmapsto f - m\cdot g_i,
$$
а значит разность лежит в идеале $(G)$ и принадлежность сохраняется.

Если редукция дала ненулевой остаток, то по произвольному $G$ **нельзя** заключить, что $f\notin (G)$: результат может зависеть от выбора редуцирующих шагов.

---

### Базис Грёбнера и критерий принадлежности
Набор $G$ — **базис Грёбнера** идеала $I=(G)$ (относительно фиксированного мономиального порядка), если идеал ведущих мономов совпадает с ведущими мономами $G$:
$$
\langle LM(I)\rangle = \langle LM(g_1),\dots,LM(g_s)\rangle.
$$
Тогда для любого $f\in R$:
$$
f\in I \iff \mathrm{NF}_G(f)=0,
$$
где $\mathrm{NF}_G(f)$ — нормальная форма (остаток) при редукции по $G$.

---

### S-полином
Для ненулевых $f,g\in R$ определим
$$
S(f,g)=\frac{\mathrm{lcm}(LM(f),LM(g))}{LT(f)}f-\frac{\mathrm{lcm}(LM(f),LM(g))}{LT(g)}g.
$$
Алгоритм Бухбергера строит базис Грёбнера, добавляя в $G$ редукции $S$-полиномов до тех пор, пока все они редуцируются в $0$.

---

### Редуцированный базис Грёбнера
Базис Грёбнера $G=\{g_1,\dots,g_s\}$ называется **редуцированным**, если:
1) ведущие коэффициенты равны $1$;
2) ведущие мономы попарно не делятся друг на друга;
3) все неначальные мономы каждого $g_i$ не делятся ни на один $LM(g_j)$, $j\ne i$.

Любой базис Грёбнера можно привести к редуцированному (алгоритмически: нормализовать, взаимно редуцировать и удалить лишние элементы). Редуцированный базис единственен с точностью до перестановки элементов.

In [9]:
Monom = Tuple[int, ...]
Coeff = Fraction
def _as_fraction(x) -> Fraction:
    if isinstance(x, Fraction):
        return x
    if isinstance(x, int):
        return Fraction(x, 1)
    if isinstance(x, float):
        return Fraction(x).limit_denominator()
    if isinstance(x, str):
        return Fraction(x)
    return Fraction(x)
def monom_mul(a: Monom, b: Monom) -> Monom:
    return tuple(ai + bi for ai, bi in zip(a, b))
def monom_divides(a: Monom, b: Monom) -> bool:
    return all(ai <= bi for ai, bi in zip(a, b))
def monom_lcm(a: Monom, b: Monom) -> Monom:
    return tuple(max(ai, bi) for ai, bi in zip(a, b))
def monom_sub(b: Monom, a: Monom) -> Monom:
    return tuple(bi - ai for ai, bi in zip(a, b))
def monom_gcd_is_one(a: Monom, b: Monom) -> bool:
    return all(min(ai, bi) == 0 for ai, bi in zip(a, b))
def key_lex(m: Monom):
    return m
def key_grlex(m: Monom):
    return (sum(m), m)
@dataclass(frozen=True)
class Poly:
    nvars: int
    terms: Dict[Monom, Coeff]
    order: str = "lex"
    def __post_init__(self):
        cleaned = {m: c for m, c in self.terms.items() if c != 0}
        object.__setattr__(self, "terms", cleaned)
        if self.order not in ("lex", "grlex"):
            raise ValueError("order must be 'lex' or 'grlex'")
    @staticmethod
    def zero(nvars: int, order: str = "lex") -> "Poly":
        return Poly(nvars, {}, order)
    @staticmethod
    def from_constant(nvars: int, c, order: str = "lex") -> "Poly":
        c = _as_fraction(c)
        return Poly.zero(nvars, order) if c == 0 else Poly(nvars, {(0,)*nvars: c}, order)
    def is_zero(self) -> bool:
        return not self.terms
    def copy(self) -> "Poly":
        return Poly(self.nvars, dict(self.terms), self.order)
    def _key(self):
        return key_lex if self.order == "lex" else key_grlex
    def leading_monomial(self) -> Monom:
        if self.is_zero():
            raise ValueError("zero polynomial has no leading monomial")
        return max(self.terms.keys(), key=self._key())
    def leading_coeff(self) -> Coeff:
        return self.terms[self.leading_monomial()]
    def leading_term(self) -> Tuple[Monom, Coeff]:
        m = self.leading_monomial()
        return m, self.terms[m]
    def monic(self) -> "Poly":
        if self.is_zero():
            return self
        lc = self.leading_coeff()
        inv = Fraction(1, 1) / lc
        return self * inv
    def __add__(self, other: "Poly") -> "Poly":
        self._check_compat(other)
        res = dict(self.terms)
        for m, c in other.terms.items():
            res[m] = res.get(m, Fraction(0)) + c
            if res[m] == 0:
                del res[m]
        return Poly(self.nvars, res, self.order)
    def __sub__(self, other: "Poly") -> "Poly":
        self._check_compat(other)
        res = dict(self.terms)
        for m, c in other.terms.items():
            res[m] = res.get(m, Fraction(0)) - c
            if res[m] == 0:
                del res[m]
        return Poly(self.nvars, res, self.order)
    def __mul__(self, other) -> "Poly":
        if isinstance(other, (int, float, Fraction)):
            c = _as_fraction(other)
            if c == 0 or self.is_zero():
                return Poly.zero(self.nvars, self.order)
            return Poly(self.nvars, {m: c*coef for m, coef in self.terms.items()}, self.order)
        if isinstance(other, Poly):
            self._check_compat(other)
            if self.is_zero() or other.is_zero():
                return Poly.zero(self.nvars, self.order)
            res: Dict[Monom, Coeff] = {}
            for m1, c1 in self.terms.items():
                for m2, c2 in other.terms.items():
                    m = monom_mul(m1, m2)
                    res[m] = res.get(m, Fraction(0)) + c1*c2
                    if res[m] == 0:
                        del res[m]
            return Poly(self.nvars, res, self.order)
        raise TypeError("unsupported multiplication")
    def mul_monom(self, m: Monom, c: Coeff = Fraction(1)) -> "Poly":
        if self.is_zero() or c == 0:
            return Poly.zero(self.nvars, self.order)
        return Poly(self.nvars, {monom_mul(m, mi): c*ci for mi, ci in self.terms.items()}, self.order)
    def _check_compat(self, other: "Poly") -> None:
        if self.nvars != other.nvars or self.order != other.order:
            raise ValueError("incompatible polynomials (nvars/order mismatch)")
    def __repr__(self) -> str:
        if self.is_zero():
            return "0"
        items = sorted(self.terms.items(), key=lambda kv: self._key()(kv[0]))
        parts = []
        for m, c in items:
            mon = []
            for i, e in enumerate(m):
                if e == 0:
                    continue
                mon.append(f"x{i+1}" if e == 1 else f"x{i+1}^{e}")
            mon_str = "*".join(mon)
            if mon_str == "":
                parts.append(str(c))
            elif c == 1:
                parts.append(mon_str)
            elif c == -1:
                parts.append("-" + mon_str)
            else:
                parts.append(f"{c}*{mon_str}")
        return " + ".join(parts).replace("+ -", "- ")
def leading_monomial(f: Poly) -> Monom:
    return f.leading_monomial()
def leading_term(f: Poly) -> Tuple[Monom, Coeff]:
    return f.leading_term()
def normal_form(f: Poly, G: List[Poly]) -> Poly:
    r = Poly.zero(f.nvars, f.order)
    p = f.copy()
    G = [g for g in G if not g.is_zero()]
    while not p.is_zero():
        lt_m, lt_c = p.leading_term()
        reduced = False
        for g in G:
            gm, gc = g.leading_term()
            if monom_divides(gm, lt_m):
                factor_m = monom_sub(lt_m, gm)
                factor_c = lt_c / gc
                p = p - g.mul_monom(factor_m, factor_c)
                reduced = True
                break
        if not reduced:
            r = r + Poly(p.nvars, {lt_m: lt_c}, p.order)
            p = p - Poly(p.nvars, {lt_m: lt_c}, p.order)
    return r
def S_polynomial(f: Poly, g: Poly) -> Poly:
    f._check_compat(g)
    lm_f, lc_f = f.leading_term()
    lm_g, lc_g = g.leading_term()
    lcm_m = monom_lcm(lm_f, lm_g)
    mul_f_m = monom_sub(lcm_m, lm_f)
    mul_g_m = monom_sub(lcm_m, lm_g)
    term1 = f.mul_monom(mul_f_m, Fraction(1,1) / lc_f)
    term2 = g.mul_monom(mul_g_m, Fraction(1,1) / lc_g)
    return term1 - term2
Criterion = Callable[[Poly, Poly], bool]
def buchberger(
    F: List[Poly],
    criteria: Optional[List[Criterion]] = None,
    use_coprime_criterion: bool = True
) -> List[Poly]:
    if not F:
        return []
    nvars, order = F[0].nvars, F[0].order
    for f in F:
        if f.nvars != nvars or f.order != order:
            raise ValueError("all polynomials must share nvars and order")
    G = [f for f in F if not f.is_zero()]
    G = [g.monic() for g in G]
    crits: List[Criterion] = []
    if use_coprime_criterion:
        def coprime_criterion(a: Poly, b: Poly) -> bool:
            return monom_gcd_is_one(a.leading_monomial(), b.leading_monomial())
        crits.append(coprime_criterion)
    if criteria:
        crits.extend(criteria)
    pairs: Set[Tuple[int,int]] = set()
    for i in range(len(G)):
        for j in range(i+1, len(G)):
            pairs.add((i,j))
    while pairs:
        i, j = pairs.pop()
        fi, fj = G[i], G[j]
        if any(c(fi, fj) for c in crits):
            continue
        s = S_polynomial(fi, fj)
        h = normal_form(s, G)
        if not h.is_zero():
            h = h.monic()
            idx_new = len(G)
            G.append(h)
            for t in range(idx_new):
                pairs.add((t, idx_new))
    return G
def reduced_groebner_basis(G: List[Poly]) -> List[Poly]:
    G = [g for g in G if not g.is_zero()]
    G = [g.monic() for g in G]
    changed = True
    while changed:
        changed = False
        keep = []
        for i, g in enumerate(G):
            lm_g = g.leading_monomial()
            if any(i != j and monom_divides(G[j].leading_monomial(), lm_g) for j in range(len(G))):
                changed = True
            else:
                keep.append(g)
        G = keep
    res = []
    for i, g in enumerate(G):
        others = [G[j] for j in range(len(G)) if j != i]
        r = normal_form(g, others)
        r = r.monic() if not r.is_zero() else r
        res.append(r)
    res = [g for g in res if not g.is_zero()]
    final = []
    for i, g in enumerate(res):
        lm_g = g.leading_monomial()
        if not any(i != j and monom_divides(res[j].leading_monomial(), lm_g) for j in range(len(res))):
            final.append(g)
    out = []
    for i, g in enumerate(final):
        others = [final[j] for j in range(len(final)) if j != i]
        out.append(normal_form(g, others).monic())
    def sort_key(p: Poly):
        k = p._key()
        return k(p.leading_monomial())
    return sorted(out, key=sort_key)
def is_in_ideal(f: Poly, G_reduced: List[Poly]) -> bool:
    if not G_reduced:
        return f.is_zero()
    nf = normal_form(f, G_reduced)
    return nf.is_zero()

In [6]:
def poly_from_terms(nvars: int, order: str, terms: Dict[Monom, int]) -> Poly:
    return Poly(nvars, {m: Fraction(c,1) for m,c in terms.items()}, order)
G0 = [
    poly_from_terms(2, "lex", {(2,0): 1}),
    poly_from_terms(2, "lex", {(1,1): 1}),
]
Gb = buchberger(G0)
Gr = reduced_groebner_basis(Gb)
assert is_in_ideal(poly_from_terms(2, "lex", {(3,0): 1}), Gr) is True
assert is_in_ideal(poly_from_terms(2, "lex", {(0,1): 1}), Gr) is False
F = [
    poly_from_terms(2, "lex", {(1,1): 1, (0,0): -1}),
    poly_from_terms(2, "lex", {(0,2): 1, (0,0): -1}),
]
Gb2 = buchberger(F)
Gr2 = reduced_groebner_basis(Gb2)
x1, x2 = sp.symbols('x1 x2')
sym_F = [x1*x2-1, x2**2-1]
sym_G = sp.groebner(sym_F, x1, x2, order='lex')
for g in Gr2:
    expr = 0
    for m,c in g.terms.items():
        expr += sp.Rational(c.numerator, c.denominator) * (x1**m[0]) * (x2**m[1])
    assert sym_G.reduce(expr)[1] == 0
f = poly_from_terms(2, "lex", {(0,1): 1, (0,0): -2})
assert is_in_ideal(f, Gr2) is False
expr = x2 - 2
assert sym_G.reduce(expr)[1] != 0
print("Expert tests passed.")

Expert tests passed.


## Практическое применение: исключение переменных (lex)

Одно из ключевых применений базиса Грёбнера: при порядке $\mathrm{lex}$ базис часто содержит полиномы, зависящие от подмножеств переменных, что позволяет последовательно решать систему (аналог исключения переменных).

Рассмотрим систему:
$$
\begin{cases}
x_1x_2-1=0,\\
x_2^2-1=0.
\end{cases}
$$
При $\mathrm{lex}$ с $x_1 \succ x_2$ из базиса Грёбнера получаем уравнение только для $x_2$, затем восстанавливаем $x_1$.

In [7]:
x1, x2 = sp.symbols('x1 x2')
G = sp.groebner([x1*x2-1, x2**2-1], x1, x2, order='lex')
G

GroebnerBasis([x1 - x2, x2**2 - 1], x1, x2, domain='ZZ', order='lex')

In [8]:
solutions = sp.solve([x1*x2-1, x2**2-1], [x1,x2], dict=True)
solutions

[{x1: -1, x2: -1}, {x1: 1, x2: 1}]